# VDJBet – Yellow Fever vaccination analysis

Replicates the `vdjbet_snippet.Rmd` analysis in Python:

1. Load LLWNGPMAV / HLA-A\*02 TRB entries from VDJdb
2. Generate a large OLGA pool and build 1 000 Pgen-matched mock reference sets
3. For each YFV donor × timepoint, count overlapping clonotypes and cells
4. Z-test against the mock distribution; plot time courses and a heatmap

In [ ]:
import math
import pickle
import warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

from mir.basic.pgen import OlgaModel
from mir.common.clonotype import Clonotype
from mir.common.parser import ClonotypeTableParser, load_vdjdb_latest
from mir.common.repertoire import LocusRepertoire
from mir.comparative.overlap import count_overlap, make_query_index, make_reference_keys

CACHE_DIR = Path("assets/large")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
YFV_DIR = CACHE_DIR / "yfv19"

## 1  Load VDJdb reference (LLWNGPMAV, HLA-A\*02, TRB)

In [ ]:
vdjdb_rep = load_vdjdb_latest(
    epitope="LLWNGPMAV",
    locus="TRB",
    species="HomoSapiens",
    mhc_a_contains="A*02",
)
print(f"Reference: {vdjdb_rep.clonotype_count} unique clonotypes")
print(f"Example: {vdjdb_rep.clonotypes[0].junction_aa}  {vdjdb_rep.clonotypes[0].v_gene}")

## 2  Generate OLGA pool (cached)

50 000 OLGA-generated TRB sequences with Pgen values.  
First run takes ~5 minutes; subsequent runs load from disk.

In [ ]:
POOL_CACHE = CACHE_DIR / "olga_trb_pool_50k.pkl"

if POOL_CACHE.exists():
    with open(POOL_CACHE, "rb") as fh:
        pool = pickle.load(fh)
    print(f"Loaded pool: {len(pool):,} sequences from cache")
else:
    print("Generating OLGA pool (one-time, ~5 min) ...")
    _model = OlgaModel(locus="TRB", seed=42)
    pool = _model.generate_sequences_with_meta(50_000, pgens=True, seed=42)
    with open(POOL_CACHE, "wb") as fh:
        pickle.dump(pool, fh)
    print(f"Pool of {len(pool):,} sequences saved to {POOL_CACHE}")

# Index pool by floor(log10(Pgen))
pool_by_bin: dict[int, list[dict]] = defaultdict(list)
for rec in pool:
    if not math.isinf(rec["pgen"]):
        pool_by_bin[int(math.floor(rec["pgen"]))].append(rec)

print(f"Bins: { {b: len(v) for b, v in sorted(pool_by_bin.items())} }")

## 3  Build 1 000 Pgen-matched mock reference sets (cached)

In [ ]:
MOCKS_CACHE = CACHE_DIR / "mock_key_sets_1000.pkl"

if MOCKS_CACHE.exists():
    with open(MOCKS_CACHE, "rb") as fh:
        mock_key_sets = pickle.load(fh)
    print(f"Loaded {len(mock_key_sets)} mock key sets from cache")
else:
    print("Computing VDJdb Pgens and generating 1 000 mock reference sets ...")
    _model = OlgaModel(locus="TRB", seed=42)

    # Compute Pgen for each VDJdb clonotype → bin
    vdjdb_bins: dict[int, list[Clonotype]] = defaultdict(list)
    for c in vdjdb_rep.clonotypes:
        p = _model.compute_pgen_junction_aa(c.junction_aa)
        if p and p > 0:
            vdjdb_bins[int(math.floor(math.log10(p)))].append(c)

    print(f"VDJdb pgen bins: { {b: len(v) for b, v in sorted(vdjdb_bins.items())} }")

    rng = np.random.default_rng(42)
    mock_key_sets: list[frozenset] = []

    for i in range(1000):
        keys: set[tuple] = set()
        for bin_val, clones in vdjdb_bins.items():
            n = len(clones)
            available = pool_by_bin.get(bin_val, [])
            if not available:
                continue
            replace = len(available) < n
            idxs = rng.choice(len(available), n, replace=replace)
            for idx in idxs:
                rec = available[idx]
                keys.add((
                    rec["junction_aa"],
                    rec["v_gene"].split("*")[0],
                    rec["j_gene"].split("*")[0],
                ))
        mock_key_sets.append(frozenset(keys))
        if (i + 1) % 200 == 0:
            print(f"  {i + 1}/1000")

    with open(MOCKS_CACHE, "wb") as fh:
        pickle.dump(mock_key_sets, fh)
    print(f"1 000 mock key sets saved to {MOCKS_CACHE}")

print(f"Example mock size: {len(mock_key_sets[0])}")

## 4  Download YFV dataset from HuggingFace

In [ ]:
if not YFV_DIR.exists() or not any(YFV_DIR.glob("*.airr.tsv.gz")):
    from huggingface_hub import snapshot_download
    print("Downloading isalgo/airr_yfv19 from HuggingFace ...")
    snapshot_download(
        repo_id="isalgo/airr_yfv19",
        repo_type="dataset",
        local_dir=str(YFV_DIR),
    )
    print(f"Downloaded to {YFV_DIR}")
else:
    print(f"Dataset already cached in {YFV_DIR}")

meta = pd.read_csv(YFV_DIR / "metadata.txt", sep="\t")
print(f"{len(meta)} samples:")
meta

## 5  Load YFV samples

In [ ]:
_parser = ClonotypeTableParser()

samples: list[dict] = []
for _, row in meta.iterrows():
    path = YFV_DIR / row["file_name"]
    df = pd.read_csv(path, sep="\t", compression="infer")
    if "locus" in df.columns:
        df = df[df["locus"].fillna("") == "TRB"]
    # Drop rows with missing or empty junction_aa before parse_inner
    # (str(NaN) would become the literal string 'nan')
    df = df.dropna(subset=["junction_aa"])
    df = df[df["junction_aa"].str.strip().str.len() > 0]
    clones = _parser.parse_inner(df)
    rep = LocusRepertoire(clonotypes=clones, locus="TRB", repertoire_id=row["file_name"])
    samples.append({
        "file_name": row["file_name"],
        "donor":     row["donor"],
        "day":       int(row["day"]),
        "replica":   str(row.get("replica", "1")),
        "repertoire": rep,
    })

print(f"Loaded {len(samples)} TRB samples")
print(f"Example: {samples[0]['donor']} day {samples[0]['day']}  "
      f"{samples[0]['repertoire'].clonotype_count:,} clonotypes")

## 6  Compute overlaps

In [ ]:
ref_keys = make_reference_keys(vdjdb_rep)
print(f"Reference keys: {len(ref_keys)}")

rows: list[dict] = []
for si, s in enumerate(samples):
    qi = make_query_index(s["repertoire"])
    n_total  = len(qi)
    dc_total = sum(qi.values())

    n_real, dc_real = count_overlap(ref_keys, qi)

    # Compute all 1000 mock overlaps in one pass to avoid iterating qi twice
    mock_results = [count_overlap(mk, qi) for mk in mock_key_sets]
    mock_n  = [r[0] for r in mock_results]
    mock_dc = [r[1] for r in mock_results]

    rows.append({
        "donor":    s["donor"],
        "day":      s["day"],
        "replica":  s["replica"],
        "n_total":  n_total,
        "dc_total": dc_total,
        "n_real":   n_real,
        "dc_real":  dc_real,
        "frac_n":   n_real  / n_total  if n_total  else 0.0,
        "frac_dc":  dc_real / dc_total if dc_total else 0.0,
        "mock_n":   mock_n,
        "mock_dc":  mock_dc,
    })
    if (si + 1) % 10 == 0:
        print(f"  {si + 1}/{len(samples)} samples processed")

df_res = pd.DataFrame(rows)
print(df_res[["donor", "day", "replica", "n_total", "n_real", "dc_real"]].to_string(index=False))

## 7  Z-test statistics (Cohen's d and p-values)

In [ ]:
def _z_p(real: float, mocks: list[float]) -> tuple[float, float]:
    arr = np.asarray(mocks, dtype=float)
    mean, std = arr.mean(), arr.std()
    if std == 0:
        return (float("inf") if real > mean else 0.0), (0.0 if real > mean else 1.0)
    z = (real - mean) / std
    return z, float(1.0 - norm.cdf(z))  # one-sided upper tail


def _stars(p: float) -> str:
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return ""


for metric, real_col, mock_col in [
    ("n",  "n_real",  "mock_n"),
    ("dc", "dc_real", "mock_dc"),
]:
    zs, ps = zip(*df_res.apply(
        lambda r: _z_p(r[real_col], r[mock_col]), axis=1
    ))
    df_res[f"z_{metric}"]     = zs
    df_res[f"p_{metric}"]     = ps
    df_res[f"stars_{metric}"] = [_stars(p) for p in ps]

df_res[["donor", "day", "replica", "z_n", "p_n", "stars_n", "z_dc", "p_dc", "stars_dc"]]

## 8  Time-course plots

In [ ]:
donors   = sorted(df_res["donor"].unique())
replicas = sorted(df_res["replica"].unique())
days_all = sorted(df_res["day"].unique())


def _box_width(days: np.ndarray) -> float:
    if len(days) < 2:
        return 3.0
    gaps = np.diff(np.sort(days))
    return float(gaps[gaps > 0].min()) * 0.4


def _plot_timecourse(real_col: str, mock_col: str, stars_col: str,
                     ylabel: str, title: str) -> None:
    fig, axes = plt.subplots(
        len(replicas), len(donors),
        figsize=(4 * len(donors), 3.5 * len(replicas)),
        squeeze=False,
        sharey=False,
    )
    fig.suptitle(title, fontsize=13, y=1.01)

    for ri, replica in enumerate(replicas):
        for di, donor in enumerate(donors):
            ax = axes[ri][di]
            sub = df_res[
                (df_res["donor"] == donor) & (df_res["replica"] == replica)
            ].sort_values("day")

            if sub.empty:
                ax.set_visible(False)
                continue

            days      = sub["day"].values
            real      = sub[real_col].values
            mock_data = list(sub[mock_col])
            width     = _box_width(days)

            ax.boxplot(
                mock_data,
                positions=days,
                widths=width,
                sym="k.",
                patch_artist=True,
                medianprops=dict(color="#888888", linewidth=1.5),
                boxprops=dict(facecolor="#d0e4f7", alpha=0.8),
                flierprops=dict(markersize=2, markerfacecolor="#888888"),
                manage_ticks=False,
            )

            ax.plot(days, real, "-o", color="#d62728",
                    linewidth=2, markersize=6, zorder=5)

            for day, rv, star in zip(days, real, sub[stars_col]):
                if star:
                    ax.text(day, rv + width * 0.1, star,
                            ha="center", va="bottom",
                            fontsize=11, color="#d62728", fontweight="bold")

            ax.set_title(f"{donor}  R{replica}", fontsize=9)
            ax.set_xlabel("Day post-vaccination", fontsize=8)
            ax.set_ylabel(ylabel, fontsize=8)
            ax.set_xticks(days_all)
            ax.tick_params(labelsize=8)

    plt.tight_layout()
    plt.show()


_plot_timecourse(
    real_col="n_real",
    mock_col="mock_n",
    stars_col="stars_n",
    ylabel="Matched clonotypes",
    title="LLWNGPMAV (HLA-A*02) matched clonotypes in YFV samples",
)

df_res["dc_real_log2"] = np.log2(df_res["dc_real"] + 1)
df_res["mock_dc_log2"] = df_res["mock_dc"].apply(
    lambda x: list(np.log2(np.array(x, dtype=float) + 1))
)

zs, ps = zip(*df_res.apply(
    lambda r: _z_p(r["dc_real_log2"], r["mock_dc_log2"]), axis=1
))
df_res["z_dc_log2"]     = zs
df_res["p_dc_log2"]     = ps
df_res["stars_dc_log2"] = [_stars(p) for p in ps]

_plot_timecourse(
    real_col="dc_real_log2",
    mock_col="mock_dc_log2",
    stars_col="stars_dc_log2",
    ylabel="log\u2082(matched cells + 1)",
    title="LLWNGPMAV (HLA-A*02) matched cells in YFV samples",
)

## 9  Heatmap: Cohen's d and significance

In [ ]:
def _heatmap(z_col: str, p_col: str, title: str,
             cmap: str = "RdBu_r", clim: tuple = (-4, 4)) -> None:
    df_res["sample_label"] = df_res["donor"] + "  R" + df_res["replica"]
    pivot_z = df_res.pivot_table(
        index="sample_label", columns="day", values=z_col, aggfunc="first"
    )
    pivot_p = df_res.pivot_table(
        index="sample_label", columns="day", values=p_col, aggfunc="first"
    )

    fig, ax = plt.subplots(figsize=(8, max(4, 0.6 * len(pivot_z))))
    im = ax.imshow(
        pivot_z.values.astype(float),
        aspect="auto", cmap=cmap,
        vmin=clim[0], vmax=clim[1],
        interpolation="nearest",
    )
    plt.colorbar(im, ax=ax, label="Cohen's d", shrink=0.8)

    for ri in range(pivot_z.shape[0]):
        for ci in range(pivot_z.shape[1]):
            p = pivot_p.values[ri, ci]
            z = pivot_z.values[ri, ci]
            if pd.notna(p) and float(p) < 0.05 and pd.notna(z) and float(z) > 0:
                ax.add_patch(plt.Rectangle(
                    (ci - 0.5, ri - 0.5), 1, 1,
                    fill=False, edgecolor="black", linewidth=2,
                ))

    ax.set_xticks(range(len(pivot_z.columns)))
    ax.set_xticklabels([f"Day {d}" for d in pivot_z.columns],
                        rotation=45, ha="right")
    ax.set_yticks(range(len(pivot_z.index)))
    ax.set_yticklabels(pivot_z.index)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


_heatmap("z_n", "p_n",
         "Clonotypes matched – Cohen's d  (black border: p < 0.05)")
_heatmap("z_dc_log2", "p_dc_log2",
         "Cells matched (log\u2082) – Cohen's d  (black border: p < 0.05)",
         cmap="Reds", clim=(0, 6))

## 10  Summary table

In [ ]:
summary = (
    df_res[[
        "donor", "day", "replica",
        "n_total", "dc_total",
        "n_real", "dc_real",
        "frac_n", "frac_dc",
        "z_n", "p_n", "stars_n",
        "z_dc", "p_dc", "stars_dc",
    ]]
    .copy()
    .sort_values(["donor", "replica", "day"])
    .reset_index(drop=True)
)

for c in ["frac_n", "frac_dc"]:
    summary[c] = summary[c].map("{:.2e}".format)
for c in ["z_n", "z_dc"]:
    summary[c] = summary[c].round(2)
for c in ["p_n", "p_dc"]:
    summary[c] = summary[c].map("{:.3f}".format)

summary.rename(columns={
    "n_total": "clonotypes", "dc_total": "cells",
    "n_real": "matched_n",   "dc_real": "matched_cells",
    "frac_n": "frac_clonotypes", "frac_dc": "frac_cells",
}, inplace=True)

pd.set_option("display.max_rows", 60)
summary